<a href="https://colab.research.google.com/github/Biegle-W/google_colab_llm/blob/cursor%2Fcolab-llm-diffusion-drive-e32d/llm_hub_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab inference hub: Qwen3.8-27B, Flash-Next, GLM-5.3-Flash, DeepSeek-V4.1-Flash-1017, Qwen-Image-2.1

One authenticated OpenAI-compatible API on a **Google AI Pro / Colab Pro** GPU for chat and images.
**Everything durable lives on Google Drive** — models, engine builds/wheels, hub package, cloudflared, config/settings, HF cache, logs, and generation outputs — so later runs restore instead of re-downloading.

- Engine picks: `colab_llm_hub/ENGINE_MATRIX.md` (your NVIDIA/AMD registries + community reports).
- Drive root: `MyDrive/llm_models/` (`models/`, `builds/`, `config/`, `logs/`, `outputs/`, `hf_cache/`, `hub/`).
- LLM targets: Qwen3.8-27B, Qwen3.8-Flash-Next, **GLM-5.3-Flash**, **DeepSeek-V4.1-Flash-1017**; plus Qwen-Image-2.1 (MiniMax-H3 prefetch for ComfyUI/SGLang).

Runtime → Change runtime type → **L4, A100 or G4**, then run the cells in order.


## 1. Mount Drive, create the workspace, fetch (or restore) the hub code

First run downloads/clones once and caches under Drive. Later runs restore the hub package, settings, and tool binaries from Drive.


In [ ]:
import os, re, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

HUB_REPO  = "https://github.com/Biegle-W/google_colab_llm"
HUB_REF   = "cursor/colab-llm-diffusion-drive-e32d"  # pin until merged; then set None for default branch
DRIVE_DIR = "/content/drive/MyDrive/llm_models"
HUB_DIR   = "/content/hub_repo"

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"{cmd}\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

os.makedirs(DRIVE_DIR, exist_ok=True)
sys.path.insert(0, HUB_DIR)
# Bootstrap: if a previous session cached the package on Drive, restore without git.
ref_key = (HUB_REF or "default").replace("/", "_")
cached = f"{DRIVE_DIR}/builds/hub_pkg/{ref_key}/colab_llm_hub"
if os.path.isdir(cached) and os.path.isfile(f"{DRIVE_DIR}/builds/hub_pkg/{ref_key}/COMPLETE"):
    shutil.rmtree(HUB_DIR, ignore_errors=True)
    os.makedirs(HUB_DIR, exist_ok=True)
    shutil.copytree(cached, f"{HUB_DIR}/colab_llm_hub")
    print("Restored hub package from Drive")
else:
    sh(f"rm -rf {HUB_DIR}")
    if HUB_REF and re.fullmatch(r"[0-9a-f]{40}", HUB_REF):
        sh(f"mkdir -p {HUB_DIR} && cd {HUB_DIR} && git init -q && git fetch -q --depth 1 {HUB_REPO} {HUB_REF} && git checkout -q FETCH_HEAD")
    else:
        sh(f"git clone -q --depth 1 {'--branch ' + HUB_REF if HUB_REF else ''} {HUB_REPO} {HUB_DIR}")

from colab_llm_hub.workspace import open_workspace
from colab_llm_hub import install
WS = open_workspace(DRIVE_DIR)
print("Drive workspace:", WS.describe())
if not (os.path.isdir(cached) and os.path.isfile(f"{DRIVE_DIR}/builds/hub_pkg/{ref_key}/COMPLETE")):
    install.cache_hub_package(HUB_DIR, str(WS.builds), ref=HUB_REF or "default")
import colab_llm_hub
print("hub", colab_llm_hub.__version__)


## 2. Detect the GPU and see which models fit it

In [ ]:
from colab_llm_hub import catalog, gpu, recommend

G = gpu.detect()[0]
VRAM = G["vram_gb"]
ENTRIES = catalog.load()
PLAN = recommend.for_gpu(VRAM, ENTRIES)
print(f"{G['name']}: {VRAM} GB VRAM, compute capability {G['compute_cap']} (sm_{G['arch']})")
!free -h | head -2
!df -h /content | tail -1
print()
print(recommend.explain(VRAM, ENTRIES))
print()
print(f"{'model id':36s} {'kind':5s} {'engine':10s} {'size':>7s}  {'ctx/edge':>14s}")
for e in ENTRIES:
    fit = catalog.fits(e, VRAM)
    kind = catalog.kind_for(e)
    label = "edge" if kind in ("image", "video") else "ctx"
    ctx = f"{catalog.context_for(e, VRAM):,} {label}" if fit else f"needs {catalog.min_vram_gb(e)} GB"
    mark = " *" if e["id"] in PLAN["prefetch"] else ""
    print(f"{e['id']:36s} {kind:5s} {e['backend']:10s} {e['size_gb']:6.1f}G  {ctx:>14s}{mark}" + ("" if fit else "   (does not fit)"))
print("\n* = recommended prefetch for this GPU (see ENGINE_MATRIX.md)")


## 3. Config (saved on Drive under `config/settings.json`)

Defaults come from the GPU recommender, then from any settings saved on Drive. Edit and re-run to persist.


In [ ]:
from colab_llm_hub import catalog, gpu, recommend

# Re-detect if this cell is run alone after a reconnect.
G = gpu.detect()[0]
VRAM = G["vram_gb"]
ENTRIES = catalog.load()
PLAN = recommend.for_gpu(VRAM, ENTRIES)
saved = WS.load_settings()

PREFETCH = saved.get("prefetch") or PLAN["prefetch"] or ["qwen3.8-27b-q4"]
ENGINES = saved.get("engines") or PLAN["engines"]
CTX_OVERRIDE = saved.get("ctx_override") or {}
LLAMA_CPP_REF = saved.get("llama_cpp_ref") or "4c3c5810f87efe177949f1b85ae509d38a2a311f"
REBUILD_LLAMA = bool(saved.get("rebuild_llama", False))
REFRESH_WHEELS = bool(saved.get("refresh_wheels", False))
TUNNEL = saved.get("tunnel") or "quick"
PORT = int(saved.get("port") or 8000)

# Persist (edit the values above, then re-run this cell to update Drive).
WS.save_settings({
    "prefetch": PREFETCH, "engines": ENGINES, "ctx_override": CTX_OVERRIDE,
    "llama_cpp_ref": LLAMA_CPP_REF, "rebuild_llama": REBUILD_LLAMA,
    "refresh_wheels": REFRESH_WHEELS, "tunnel": TUNNEL, "port": PORT,
    "hub_ref": HUB_REF, "gpu_name": G["name"], "vram_gb": VRAM, "band": PLAN["band"],
})
print("Saved settings →", WS.settings_path)
print("PREFETCH =", PREFETCH)
print("ENGINES  =", ENGINES)
unknown = [m for m in PREFETCH if m not in {e["id"] for e in ENTRIES}]
assert not unknown, f"unknown model ids: {unknown}"
not_fit = [m for m in PREFETCH if not catalog.fits(catalog.by_id(ENTRIES)[m], VRAM)]
assert not not_fit, f"these do not fit this GPU: {not_fit}"
needed = {catalog.by_id(ENTRIES)[m]["backend"] for m in PREFETCH}
assert needed <= set(ENGINES), f"PREFETCH needs engines {sorted(needed - set(ENGINES))}; add them to ENGINES"


## 4. Install the engines (built or downloaded once, then restored from Drive)

In [ ]:
from colab_llm_hub import install

BUILDS = str(WS.builds)
LLAMA_BIN = "llama-server"
install.ensure_hub_deps(BUILDS, refresh=REFRESH_WHEELS)
if "llama" in ENGINES:
    LLAMA_BIN = install.ensure_llama(BUILDS, G["arch"], ref=LLAMA_CPP_REF, rebuild=REBUILD_LLAMA)
    print("llama-server:", LLAMA_BIN)
if "vllm" in ENGINES:
    print("vLLM", install.ensure_vllm(BUILDS, refresh=REFRESH_WHEELS))
if "sglang" in ENGINES:
    print("SGLang", install.ensure_sglang(BUILDS, refresh=REFRESH_WHEELS))
if "diffusers" in ENGINES:
    print("diffusers", install.ensure_diffusers(BUILDS, refresh=REFRESH_WHEELS))
print("Install log →", WS.logs / "install.log")


## 5. Download the models to Google Drive (skipped for anything already saved)


In [ ]:
import threading, time
from colab_llm_hub.store import ModelStore

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

store = ModelStore(DRIVE_DIR, str(WS.local_models))
for mid in PREFETCH:
    e = catalog.by_id(ENTRIES)[mid]
    if store.state(e)["on_drive"]:
        print(f"{mid}: already on Drive")
        continue
    stop = threading.Event()
    def report(mid=mid, e=e):
        while not stop.wait(15):
            p = store.progress.get(mid, {})
            print(f"  {mid}: {p.get('phase', '')} {p.get('done_bytes', 0) / 1e9:.1f} / {e['size_gb']:.1f} GB")
    threading.Thread(target=report, daemon=True).start()
    t0 = time.time()
    try:
        store.prefetch(e)
    finally:
        stop.set()
    print(f"{mid}: saved to Drive in {(time.time() - t0) / 60:.1f} min")


## 6. Start the hub and open the tunnel

Re-running this cell restarts the hub. The API key is created once and kept on Drive, so your harness settings stay valid across sessions.

In [ ]:
import json, requests, time
from colab_llm_hub import keyfile, tunnel, provider

subprocess.run("pkill -TERM -f '[c]olab_llm_hub serve'; pkill -f '[c]loudflared'", shell=True)
time.sleep(3)

API_KEY = keyfile.load_or_create(str(WS.hub / "api_key"))
HUB_LOG = str(WS.logs / "hub.log")
args = [sys.executable, "-m", "colab_llm_hub", "serve", "--drive-dir", DRIVE_DIR,
        "--local-dir", str(WS.local_models), "--log-dir", str(WS.engine_logs),
        "--port", str(PORT), "--llama-bin", LLAMA_BIN, "--engines", ",".join(ENGINES)]
for k, v in CTX_OVERRIDE.items():
    args += ["--ctx", f"{k}={v}"]
hub_proc = subprocess.Popen(args, cwd=HUB_DIR, env=dict(os.environ, PYTHONPATH=HUB_DIR, COLAB_LLM_DRIVE=DRIVE_DIR),
                            stdout=open(HUB_LOG, "w"), stderr=subprocess.STDOUT)

AUTH = {"Authorization": f"Bearer {API_KEY}"}
for _ in range(60):
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", headers=AUTH, timeout=2).status_code == 200:
            break
    except requests.RequestException:
        pass
    if hub_proc.poll() is not None:
        raise RuntimeError("The hub exited:\n" + open(HUB_LOG).read()[-3000:])
    time.sleep(1)
else:
    raise RuntimeError("The hub did not start:\n" + open(HUB_LOG).read()[-3000:])

cf = tunnel.ensure_cloudflared(drive_dir=DRIVE_DIR)
if TUNNEL == "named":
    from google.colab import userdata
    tunnel_proc = tunnel.start_named(userdata.get("CLOUDFLARE_TUNNEL_TOKEN"), drive_dir=DRIVE_DIR)
    BASE = "https://" + userdata.get("CLOUDFLARE_HOSTNAME").removeprefix("https://").strip("/")
else:
    tunnel_proc, BASE = tunnel.start_quick(PORT, drive_dir=DRIVE_DIR)

MODELS = requests.get(f"http://127.0.0.1:{PORT}/v1/models", headers=AUTH).json()["data"]
WS.save_endpoint(BASE + "/v1", API_KEY, G["name"], [m["id"] for m in MODELS])

print("Base URL :", BASE + "/v1")
print("API key  :", API_KEY)
print(f"Models   : {', '.join(m['id'] for m in MODELS)}")
print("Endpoint →", WS.hub / "endpoint.json")
print()
print("Any OpenAI-compatible client:")
print(provider.openai_env(BASE, API_KEY))
print("dsh: on your computer, in a clone of this repo:")
print(f"  python -m colab_llm_hub.configure_dsh --url {BASE} --key {API_KEY} --settings <path to dsh settings.yaml>")
print(f"  export COLAB_LLM_API_KEY={API_KEY}\n")
print(provider.dsh_provider_yaml(BASE, MODELS))


## 7. Try chat and image generation

The first request to a model loads it (download or Drive→local copy, then GPU load). Streaming chat waits with keep-alives; non-streaming may return `503` + `Retry-After` while loading — re-run the cell.


In [ ]:
from IPython.display import display, Image as IPyImage
import base64

chat_models = [m for m in MODELS if m.get("kind") != "image"]
image_models = [m for m in MODELS if m.get("kind") == "image"]

if chat_models:
    MODEL = next((m for m in PREFETCH if m in {x["id"] for x in chat_models}), chat_models[0]["id"])
    messages = [{"role": "user", "content": "In two sentences, what is a model hub?"}]
    t0 = time.time()
    r = requests.post(f"http://127.0.0.1:{PORT}/v1/chat/completions", headers=AUTH, timeout=900,
                      json={"model": MODEL, "max_tokens": 200, "messages": messages})
    if r.status_code == 503 and r.json().get("error", {}).get("code") == "model_loading":
        print("Chat model still loading; re-run this cell in a minute.")
    else:
        j = r.json()
        text = j["choices"][0]["message"]["content"] if r.ok else j
        print("=== chat:", MODEL, "===")
        print(text)
        print(f"[{time.time() - t0:.1f} s]")
        if r.ok:
            path = WS.save_chat(MODEL, messages, text, j.get("usage"))
            print("Saved →", path)

if image_models:
    IMG = next((m for m in PREFETCH if m in {x["id"] for x in image_models}), image_models[0]["id"])
    prompt = "a watercolor fox under pine trees, soft light"
    t0 = time.time()
    r = requests.post(f"http://127.0.0.1:{PORT}/v1/images/generations", headers=AUTH, timeout=900,
                      json={"model": IMG, "prompt": prompt, "size": "768x768", "n": 1})
    if r.status_code == 503 and r.json().get("error", {}).get("code") == "model_loading":
        print("Image model still loading; re-run this cell in a minute.")
    else:
        j = r.json()
        print("=== image:", IMG, j.get("size"), "===")
        if r.ok:
            display(IPyImage(data=base64.b64decode(j["data"][0]["b64_json"])))
            path = WS.save_image_b64(j["data"][0]["b64_json"], IMG, prompt=prompt)
            print(f"[{time.time() - t0:.1f} s] Saved →", path)
        else:
            print(j)


## 8. Status and logs (re-run any time)

In [ ]:
st = requests.get(f"http://127.0.0.1:{PORT}/v1/hub/status", headers=AUTH).json()
WS.save_json(WS.status, "hub-status", st)
print(json.dumps({k: st[k] for k in ("gpu", "vram_gb", "active", "loading", "phase", "inflight", "errors", "local_free_gb")}, indent=1))
for mid, p in st["downloads"].items():
    print(f"  {mid}: {p.get('phase')} {p.get('done_bytes', 0) / 1e9:.1f} / {p.get('total_bytes', 0) / 1e9:.1f} GB")
print("\n--- hub log (Drive) ---"); print(open(str(WS.logs / "hub.log")).read()[-1500:])
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
print("Outputs →", WS.outputs)


## 9. Stop

In [ ]:
hub_proc.terminate()        # stops the hub and the engine it started
tunnel_proc.terminate()

## Notes

### Drive layout (nothing important should live only under `/content`)

| Path under `MyDrive/llm_models` | Contents |
|---|---|
| `models/<id>/` | Weights (GGUF / safetensors / Diffusers), marked complete only after a full copy |
| `builds/` | llama.cpp tarballs, vLLM/SGLang/Diffusers/hub_deps wheels, cached `hub_pkg/`, `tools/cloudflared` |
| `config/settings.json` | Prefetch list, engines, context overrides, tunnel/port |
| `hub/api_key`, `hub/endpoint.json` | Stable API key + last public URL |
| `hf_cache/` | Hugging Face / Diffusers / Transformers cache (`HF_HOME`) |
| `logs/` | `install.log`, `hub.log`, `cloudflared.log`, `hub_logs/` |
| `outputs/images/`, `outputs/chat/`, `outputs/status/` | Generation results and status snapshots |

### Target models (engine picks)
See `colab_llm_hub/ENGINE_MATRIX.md`.

| Model | Colab path | Your workstation |
|---|---|---|
| Qwen3.8-27B | vLLM W4A16 / FP8, SGLang, llama Q4 | HyperQwen / SGLang |
| Qwen3.8-Flash-Next | llama.cpp IQ1_S / Q3 + n-gram CPU | Strata |
| **GLM-5.3-Flash** | llama.cpp UD-IQ1_S (~93 GB; needs GLM-capable build) | **Maya-S24** |
| **DeepSeek-V4.1-Flash-1017** | llama.cpp UD-IQ1_S (~233 GB Drive + Engram mmap) | **Maya / Strata** |
| Qwen-Image-2.1 | Diffusers NF4 / BF16 | ComfyUI Turbo |
| MiniMax-H3 | Prefetch only → ComfyUI / SGLang diffusion | ComfyUI |

- **Google AI Pro / Colab Pro:** Runtime → L4, A100 or G4. GLM/DeepSeek want A100 high-RAM or G4.
- Large MoE GGUFs may need a newer / Unsloth llama.cpp than the default Qwen pin — set `LLAMA_CPP_REF` / rebuild when those models fail to load.
- Set `REFRESH_WHEELS` / `REBUILD_LLAMA` in settings to force updates.
- **Security:** tunnel URL is public; only the API key protects GPU use.
